In [1]:
import torch

In [ ]:
T = 8
pad_mask = torch.ones((T, T), dtype=torch.bool)
rand_index = 0


In [ ]:
T = 8
chunk_size = 32
outputs = torch.randn((1, 2, chunk_size))


torch.Size([1, 2, 32])

In [4]:
x = torch.arange(32)
x

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [5]:
x_padded = torch.nn.functional.pad(x, (7, 0), value=-1)
x_padded

tensor([-1, -1, -1, -1, -1, -1, -1,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10,
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28,
        29, 30, 31])

In [7]:
batches = x_padded.unfold(0, 8, 1)
batches.shape

torch.Size([32, 8])

In [17]:
L = 8 # Block size

# 1. Create the Causal Mask (Standard 8x8 triangle)
# True = Mask out (Future)
causal_mask = torch.triu(torch.ones(L, L), diagonal=1).bool()

# 2. Create the Padding Mask (Dynamic for each batch)
# This looks at your batches and finds where the values are 0
# Result shape: [32, 8]
padding_mask = (batches == -1)

# 3. Combine them into a 3D mask: [32, 8, 8]
# We unsqueeze the padding mask so it applies to the 'Key' dimension
# Logic: Mask if it's the Future OR if the Key is a Padding token
final_mask = causal_mask.unsqueeze(0) | padding_mask.unsqueeze(1)

In [21]:
causal_mask[0]

tensor([False,  True,  True,  True,  True,  True,  True,  True])

In [20]:
final_mask[3]

tensor([[ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False,  True,  True,  True],
        [ True,  True,  True,  True, False, False,  True,  True],
        [ True,  True,  True,  True, False, False, False,  True],
        [ True,  True,  True,  True, False, False, False, False]])